# Parametrized distribution re-run for v1-augmented provinces

Generalizes notebook 09 (Iloilo re-run) to any list of provinces. For each target province, drop existing `bus_type='distribution'` rows and regenerate using **all** current substation-class roots (`substation`, `substation_synth`, `generator`, `hvdc`, `bess`). Other provinces' distribution is untouched.

**Default target list** = every province with at least one v1-imported bus, except Iloilo (already done in notebook 09):
Aklan, Bohol, Capiz, Cebu, Guimaras, Leyte, Negros Occidental, Negros Oriental.

**Same parameters as Phase 1C / notebook 09:** `N_FEEDERS_PER_ROOT=4`, `BRANCHES_PER_FEEDER=6`, `LOAD_PER_BUS_MW=1.5`, `FEEDER_MIN_KM=5`, `FEEDER_MAX_KM=25`. Per-province deterministic seed = `20260513 + i` where `i` is the province's alphabetical index in the target list — different from Phase 1C (`20260511`) and notebook 09 (`20260512`) to avoid bus_id collisions on re-run.

**Known side effect:** total Visayas synthetic peak load will increase, because every v1-imported substation now seeds 24 more distribution buses × 1.5 MW. The Cebu overshoot (already 1.2 GW vs ~500 MW real) will get worse before per-province root-count tuning lands.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

PROC_DIR  = Path('../backend/data/processed')
BOUND_DIR = Path('../backend/data/boundaries')
WGS = 'EPSG:4326'
UTM = 'EPSG:32651'

DIST_VOLTAGE_KV     = 13.8
N_FEEDERS_PER_ROOT  = 4
BRANCHES_PER_FEEDER = 6
LOAD_PER_BUS_MW     = 1.5
PF                  = 0.9
TAN_PHI             = math.tan(math.acos(PF))

DIST_R_OHM_PER_KM = 0.40
DIST_X_OHM_PER_KM = 0.40
DIST_MAX_I_KA     = 0.30

FEEDER_MIN_KM    = 5.0
FEEDER_MAX_KM    = 25.0
MAX_REJECT_TRIES = 200

ROOT_TYPES = ['substation', 'substation_synth', 'generator', 'hvdc', 'bess']

TARGET_PROVINCES = sorted([
    'Aklan', 'Bohol', 'Capiz', 'Cebu', 'Guimaras',
    'Leyte', 'Negros Occidental', 'Negros Oriental',
])  # Iloilo intentionally excluded — already done in notebook 09
SEED_BASE = 20260513

print('Targets:', TARGET_PROVINCES)

Targets: ['Aklan', 'Bohol', 'Capiz', 'Cebu', 'Guimaras', 'Leyte', 'Negros Occidental', 'Negros Oriental']


In [2]:
buses = pd.read_csv(PROC_DIR / 'buses.csv')
lines = pd.read_csv(PROC_DIR / 'lines.csv')
provinces = gpd.read_file(BOUND_DIR / 'psgc_provinces.geojson')
print(f'Inputs: {len(buses)} buses, {len(lines)} lines')

Inputs: 2522 buses, 2529 lines


## §1 — Pre-run snapshot per target province

In [3]:
pre_snapshot = []
for pname in TARGET_PROVINCES:
    p_rows = buses[buses['province'] == pname]
    n_roots = len(p_rows[p_rows['bus_type'].isin(ROOT_TYPES)])
    n_dist  = len(p_rows[p_rows['bus_type'] == 'distribution'])
    load_mw = p_rows['p_mw'].sum()
    pre_snapshot.append({
        'province': pname,
        'roots': n_roots,
        'dist_buses_before': n_dist,
        'load_mw_before': load_mw,
    })
pre_df = pd.DataFrame(pre_snapshot)
print(pre_df.to_string(index=False))

         province  roots  dist_buses_before  load_mw_before
            Aklan      2                 24            36.0
            Bohol     11                186           279.0
            Capiz      1                  0             0.0
             Cebu     42                812          1218.0
         Guimaras      1                  0             0.0
            Leyte     20                364           546.0
Negros Occidental     13                226           339.0
  Negros Oriental     12                172           258.0


## §2 — Phase 1C feeder generator (verbatim from notebooks 03 + 09)

In [4]:
def random_point_in_polygon(poly, around_point_m, min_dist_m, max_dist_m, bearing_rad, rng):
    wedge_half_width = math.radians(35)
    for _ in range(MAX_REJECT_TRIES):
        d = rng.uniform(min_dist_m, max_dist_m)
        theta = rng.uniform(bearing_rad - wedge_half_width, bearing_rad + wedge_half_width)
        x = around_point_m.x + d * math.cos(theta)
        y = around_point_m.y + d * math.sin(theta)
        p = Point(x, y)
        if poly.contains(p):
            return p
    return None

def synthesize_feeders_for_root(root_row, province_poly_m, n_feeders, branches, rng):
    root_pt_m = gpd.GeoSeries(
        [Point(root_row['lon'], root_row['lat'])], crs=WGS
    ).to_crs(UTM).iloc[0]
    bus_rows, line_rows = [], []
    bearings = np.linspace(0, 2 * math.pi, n_feeders, endpoint=False)
    bearings += rng.uniform(0, 2 * math.pi / n_feeders)
    for f_idx, bearing in enumerate(bearings):
        prev_bus_id = root_row['bus_id']
        prev_pt = root_pt_m
        for b_idx in range(branches):
            min_d = FEEDER_MIN_KM * 1000 * (b_idx + 1) / branches
            max_d = FEEDER_MAX_KM * 1000 * (b_idx + 1) / branches
            pt = random_point_in_polygon(province_poly_m, root_pt_m, min_d, max_d, bearing, rng)
            if pt is None:
                break
            new_bus_id = f'{root_row["bus_id"]}_F{f_idx}_B{b_idx}'
            bus_rows.append({
                '_pt_m': pt,
                'bus_id': new_bus_id, 'name': new_bus_id,
                'voltage_kv': DIST_VOLTAGE_KV,
                'province': root_row['province'], 'island': root_row['island'],
                'bus_type': 'distribution',
                'p_mw': LOAD_PER_BUS_MW, 'q_mvar': LOAD_PER_BUS_MW * TAN_PHI,
                'is_synthetic': True, 'data_source': 'synthetic',
            })
            line_rows.append({
                'from_bus': prev_bus_id, 'to_bus': new_bus_id,
                'voltage_kv': DIST_VOLTAGE_KV,
                'length_km':  prev_pt.distance(pt) / 1000,
                'r_ohm_per_km': DIST_R_OHM_PER_KM, 'x_ohm_per_km': DIST_X_OHM_PER_KM,
                'max_i_ka': DIST_MAX_I_KA,
                'is_submarine': False, 'cable_type': 'overhead',
                'is_synthetic': True, 'data_source': 'synthetic',
            })
            prev_bus_id = new_bus_id
            prev_pt = pt
    return bus_rows, line_rows

## §3 — Drop old distribution and regenerate per province

In [5]:
# Drop existing distribution buses in target provinces, and any lines incident on them
drop_mask = (
    buses['province'].isin(TARGET_PROVINCES) &
    (buses['bus_type'] == 'distribution')
)
drop_bus_ids = set(buses.loc[drop_mask, 'bus_id'])
buses_kept = buses[~buses['bus_id'].isin(drop_bus_ids)].copy()
lines_kept = lines[
    ~lines['from_bus'].isin(drop_bus_ids) & ~lines['to_bus'].isin(drop_bus_ids)
].copy()
print(f'Dropped {len(drop_bus_ids)} distribution buses and '
      f'{len(lines) - len(lines_kept)} incident lines across {len(TARGET_PROVINCES)} provinces.')

Dropped 1784 distribution buses and 1784 incident lines across 8 provinces.


In [6]:
all_new_buses, all_new_lines, per_province = [], [], []
for i, pname in enumerate(TARGET_PROVINCES):
    RNG = np.random.default_rng(seed=SEED_BASE + i)
    poly_m = provinces[provinces['province'] == pname].to_crs(UTM).iloc[0].geometry
    roots = buses_kept[
        (buses_kept['province'] == pname) &
        (buses_kept['bus_type'].isin(ROOT_TYPES))
    ]
    p_buses, p_lines = [], []
    for _, root in roots.iterrows():
        b, l = synthesize_feeders_for_root(
            root, poly_m, N_FEEDERS_PER_ROOT, BRANCHES_PER_FEEDER, RNG)
        p_buses.extend(b); p_lines.extend(l)
    all_new_buses.extend(p_buses); all_new_lines.extend(p_lines)
    per_province.append({
        'province': pname,
        'roots': len(roots),
        'new_dist_buses': len(p_buses),
        'new_dist_lines': len(p_lines),
        'new_load_mw': len(p_buses) * LOAD_PER_BUS_MW,
    })
    print(f'  {pname:<20s} {len(roots):2d} roots → {len(p_buses):4d} dist buses, '
          f'{len(p_buses)*LOAD_PER_BUS_MW:6.1f} MW')

summary_df = pd.DataFrame(per_province)
print(f'\nTotal new across {len(TARGET_PROVINCES)} provinces: '
      f'{len(all_new_buses)} dist buses, {len(all_new_lines)} lines, '
      f'{len(all_new_buses)*LOAD_PER_BUS_MW:.1f} MW')

  Aklan                 2 roots →   44 dist buses,   66.0 MW
  Bohol                11 roots →  235 dist buses,  352.5 MW
  Capiz                 1 roots →   24 dist buses,   36.0 MW


  Cebu                 42 roots →  922 dist buses, 1383.0 MW
  Guimaras              1 roots →   20 dist buses,   30.0 MW
  Leyte                20 roots →  443 dist buses,  664.5 MW
  Negros Occidental    13 roots →  290 dist buses,  435.0 MW


  Negros Oriental      12 roots →  244 dist buses,  366.0 MW

Total new across 8 provinces: 2222 dist buses, 2222 lines, 3333.0 MW


## §4 — Re-project, align schema, append, save

In [7]:
new_buses_m = gpd.GeoDataFrame(
    all_new_buses, geometry=[r['_pt_m'] for r in all_new_buses], crs=UTM
).to_crs(WGS)
new_buses_df = pd.DataFrame({
    'bus_id':       new_buses_m['bus_id'],
    'name':         new_buses_m['name'],
    'lat':          new_buses_m.geometry.y,
    'lon':          new_buses_m.geometry.x,
    'voltage_kv':   new_buses_m['voltage_kv'],
    'province':     new_buses_m['province'],
    'island':       new_buses_m['island'],
    'bus_type':     new_buses_m['bus_type'],
    'p_mw':         new_buses_m['p_mw'],
    'q_mvar':       new_buses_m['q_mvar'],
    'is_synthetic': new_buses_m['is_synthetic'],
    'data_source':  new_buses_m['data_source'],
})

new_lines_df = pd.DataFrame(all_new_lines)
new_lines_df['line_id'] = [f'line_synth_redist_{i:05d}' for i in range(len(new_lines_df))]

buses_out = pd.concat([buses_kept, new_buses_df.reindex(columns=buses_kept.columns, fill_value=np.nan)], ignore_index=True)
lines_out = pd.concat([lines_kept, new_lines_df.reindex(columns=lines_kept.columns, fill_value=np.nan)], ignore_index=True)

assert buses_out['bus_id'].is_unique
assert lines_out['line_id'].is_unique
valid = set(buses_out['bus_id'])
assert lines_out['from_bus'].isin(valid).all() and lines_out['to_bus'].isin(valid).all()

buses_out.to_csv(PROC_DIR / 'buses.csv', index=False)
lines_out.to_csv(PROC_DIR / 'lines.csv', index=False)
summary_df.to_csv(PROC_DIR / 'redistribute_summary.csv', index=False)
print(f'Wrote buses.csv ({len(buses_out)} rows)')
print(f'Wrote lines.csv ({len(lines_out)} rows)')
print(f'Wrote redistribute_summary.csv ({len(summary_df)} rows)')

Wrote buses.csv (2960 rows)
Wrote lines.csv (2967 rows)
Wrote redistribute_summary.csv (8 rows)


## §5 — Before/after comparison + global state

In [8]:
post_snapshot = []
for pname in TARGET_PROVINCES:
    p_rows = buses_out[buses_out['province'] == pname]
    n_dist = len(p_rows[p_rows['bus_type'] == 'distribution'])
    load_mw = p_rows['p_mw'].sum()
    post_snapshot.append({
        'province': pname,
        'dist_buses_after': n_dist,
        'load_mw_after': load_mw,
    })
post_df = pd.DataFrame(post_snapshot)

compare = pre_df.merge(post_df, on='province')
compare['dist_delta'] = compare['dist_buses_after'] - compare['dist_buses_before']
compare['load_delta_mw'] = compare['load_mw_after'] - compare['load_mw_before']
print('Per-province before vs after:')
print(compare[['province', 'roots', 'dist_buses_before', 'dist_buses_after', 'dist_delta',
                'load_mw_before', 'load_mw_after', 'load_delta_mw']].to_string(index=False))

total_load_after = buses_out['p_mw'].sum()
print(f'\nTotal Visayas synthetic peak load: {total_load_after:.0f} MW')
print(f'(real Visayas peak is ~2 200 MW)')

Per-province before vs after:
         province  roots  dist_buses_before  dist_buses_after  dist_delta  load_mw_before  load_mw_after  load_delta_mw
            Aklan      2                 24                44          20            36.0           66.0           30.0
            Bohol     11                186               235          49           279.0          352.5           73.5
            Capiz      1                  0                24          24             0.0           36.0           36.0
             Cebu     42                812               922         110          1218.0         1383.0          165.0
         Guimaras      1                  0                20          20             0.0           30.0           30.0
            Leyte     20                364               443          79           546.0          664.5          118.5
Negros Occidental     13                226               290          64           339.0          435.0           96.0
  Negros O

In [9]:
# Component check
import networkx as nx
G = nx.Graph()
G.add_nodes_from(buses_out['bus_id'])
G.add_edges_from(zip(lines_out['from_bus'], lines_out['to_bus']))
ncc = nx.number_connected_components(G)
sizes = sorted([len(c) for c in nx.connected_components(G)], reverse=True)
print(f'Whole-grid components: {ncc}')
print(f'Top 5 component sizes: {sizes[:5]}')

Whole-grid components: 69
Top 5 component sizes: [1178, 225, 206, 50, 44]
